# N055 · 单调栈的面积与贡献计数

**目标：** 从元素左右作用范围推导全局统计量。

**先修：** N054, N031。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 直方图边界；哨兵；重复值去重归属；子数组贡献；模数。

**配套讲解来源：** [同名逐章意见](../../comment/055_monotonic_stack_contributions.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

上一章（N054）单调栈回答的是"每个元素旁边的邻居是谁"；这一章更进一层：**用每个元素的"管辖范围"去算一个全局的统计量**。两个代表题：

1. **直方图中最大的矩形**：一排宽度都是 1 的柱子给你高度，问能框出的最大矩形面积。输入 `[2,1,5,6,2,3]` → 输出 10。为什么：高度 5 和 6 的两根柱子并排（下标 2、3），被高度 5 限制住，能框出 5×2=10 的矩形；虽然 6 更高，但它左边只有 5，右边降到 2，单独一根只能 6×1=6。思路是**枚举"哪根柱子是矩形的高度上限"**：固定某根柱 j 当最低高度，它能往左右延伸到"最近的一根比它矮的柱子"为止。
2. **所有子数组的最小值之和**：把数组所有连续子数组的 min 加起来。输入 `[3,1,2,4]` → 输出 17。为什么：全部 10 个子数组是 `[3],[3,1],[3,1,2],[3,1,2,4],[1],[1,2],[1,2,4],[2],[2,4],[4]`，各自的最小值 3,1,1,1,1,1,1,2,2,4，加起来 17。暴力要枚举所有 O(n²) 个子数组；贡献法换一个角度——**问每个元素"有多少个子数组的最小值是你"**，一次扫描就算完。

两题共享同一个骨架：单调栈在弹栈的一瞬间，正好告诉你"被弹元素的右边界刚刚出现"，而栈顶下面那个就是它的左边界。

先看暴力有多贵：最大矩形若枚举所有区间 `[l, r)` 再取 `min(heights[l:r])*(r-l)`，子区间有约 n²/2 个、每个取 min 又要 O(n)，总共 O(n³)；最小值和同理要 O(n²) 个子数组。n=10⁵ 时连 O(n²) 都是百亿级，必须换思路。本章的换法在哲学上很典型：**不枚举"答案的容器"（子数组/矩形），改枚举"答案的瓶颈"（最小值是哪根柱、最低柱是哪根），每个瓶颈只结算一次**。

## 2. 基础知识：先读懂这些词

- **管辖范围（左右作用范围）**：元素 j 作为"最小值/最低柱"时，它能覆盖的最宽区间——从"左边最近的更小元素"的下一格，到"右边最近的更小元素"的前一格。范围内的所有子数组/矩形都以 j 为瓶颈。
- **直方图边界**：最大矩形题里，柱 j 的左边界是左侧最近的**更矮**柱，右边界是右侧最近的更矮柱；宽度 = 右边界下标 − 左边界下标 − 1（两头都不算），面积 = heights[j] × 宽度。
- **哨兵（sentinel）**：人为加到数组末尾的一个"必定触发弹栈"的值。直方图版加一个高度 0 的虚拟柱，最小值版用 `i==len(nums)` 这个条件充当哨兵，它们保证扫描结束时栈里所有元素都被清算，不需要循环后再写一段收尾代码。
- **贡献法（contribution counting）**：不求"每个子数组的 min"，而是反过来问"元素 j 是多少个子数组的 min"。设 j 的左边界是 left（严格更小的最近元素下标）、右边界是 i，那么左端点可选 (j−left) 种、右端点可选 (i−j) 种，j 的贡献就是 `nums[j]×(j−left)×(i−j)`。
- **重复值去重归属**：当子数组里最小值出现多次时，必须规定"只归一个人"，否则会重复计数。本章的规则（“正确性与复杂度”部分）：归**最右边那次出现**——左边界取严格更小（相等的邻居不算边界），右边界取小于等于（相等的也会触发弹栈）。一侧严格、一侧非严格，是"既不重也不漏"的关键。
- **模数（mod，10⁹+7）**：最小值之和可能非常大，题目约定输出对 10⁹+7 取模的余数。小数组测试时结果不取模也能对上（余数就是本身）。

## 3. 思路推导：从小例子开始

**largest_rectangle 的步骤：**

1. 在高度数组末尾拼一个 0（哨兵），从左到右扫。
2. 新柱 h 到来时，栈里所有**比 h 高**的柱子可以被结算了——因为 h 就是它们的右边界（第一根更矮的柱）。
3. 结算弹出的柱 j：它的新栈顶（弹出后露出来的）就是左边最近的更矮柱 left（栈空则记 -1），宽度 `i-left-1`，面积 `heights[j]*宽度`，更新最优。
4. 把当前下标压栈，继续。

**手算演示：`[2,1,5,6,2,3]`（拼哨兵后扫 7 步）——**

| i | 高度 h | 弹出的 j | left | 宽度 i−left−1 | 面积 | best |
|---|--------|----------|------|----------------|------|------|
| 0 | 2 | — | — | — | — | 0 |
| 1 | 1 | 0（高2） | -1 | 1 | 2 | 2 |
| 2 | 5 | — | — | — | — | 2 |
| 3 | 6 | — | — | — | — | 2 |
| 4 | 2 | 3（高6） | 2 | 1 | 6 | 6 |
|   |   | 2（高5） | 1 | 2 | **10** | **10** |
| 5 | 3 | — | — | — | — | 10 |
| 6 | 0(哨兵) | 5,4,1 依次弹出 | | | 3,8,6 | 10 |

i=4 时 2 一来，把 6 和 5 连续弹出，5 那次结算得到 5×2=10，这就是全局最优；哨兵 0 最后把剩在栈里的柱子全部清账。

**sum_subarray_mins 的步骤：**

1. 扫描下标 i 从 0 到 n（n 是"越界位置"，充当哨兵）。
2. 触发弹栈的条件是 `nums[栈顶] >= nums[i]`（注意是**大于等于**——相等的也弹，这把最小值归属权交给最右边的出现者）。
3. 每弹出 j：left = 弹出后的新栈顶（没有则 -1）。因为相等的早被弹走，left 一定是**严格小于** nums[j] 的最近者。贡献 `nums[j]*(j-left)*(i-j)` 累加。
4. 全部结束后对 10⁹+7 取模。

**手算演示（“运行示例”部分 的表格，a=[3,1,2,4]）：**

| 负责索引 j | 值 | 左严格小边界 left | 右小于等于边界 i | 区间数 (j−left)(i−j) | 贡献 |
|-----------|----|--------------------|-------------------|----------------------|------|
| 0 | 3 | -1 | 1 | 1×1=1 | 3 |
| 3 | 4 | 2 | 4 | 1×1=1 | 4 |
| 2 | 2 | 1 | 4 | 1×2=2 | 4 |
| 1 | 1 | -1 | 4 | 2×3=6 | 6 |

合计 3+4+4+6=17，和逐个子数组暴力相加的结果一致。看第 4 行：值 1 的管辖范围是整个数组（左边没有更小的，右边到哨兵为止），左端点可选 {0,1} 共 2 种（j−left=1−(−1)=2），右端点可选 {1,2,3} 共 3 种（i−j=3），所以 6 个以它为最小值的子数组，贡献 1×6=6。

再补一个含重复值的迷你演示，看看"归最右出现者"怎么落账：`nums=[1,1]`（k 任意）。子数组三个：`[1]`、`[1]`、`[1,1]`，最小值和 = 3。跑算法：i=0 压栈；i=1 时值 1 满足 `nums[0]>=1` 弹出 j=0，left=-1，贡献 `1*(0-(-1))*(1-0)=1`（第一个 1 只负责 `[第一个1]` 这一个子数组）；哨兵 i=2 时弹出 j=1，left=-1（相等的前一个已被弹走，栈空），贡献 `1*(1-(-1))*(2-1)=2`（第二个 1 负责 `[第二个1]` 和 `[1,1]` 两个子数组）。总计 1+2=3，分毫不差——相等的一对，靠"右侧非严格弹栈"把跨两个的子数组判给了右边那位。

**重复值的几何直觉**：把一排相等的柱子想象成一排同高的人，"谁负责跨过全体的子数组"必须指定一个人，本章指定队尾（最右）那位。左侧用严格比较是为了不把左边的相等者也当成边界（否则右边这位的管辖范围会被误砍一刀）；右侧用非严格比较是为了让左边那位及时出局（否则跨全体的子数组会被左右两边各算一次）。

## 4. 核心代码：largest_rectangle

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def largest_rectangle(heights):
    stack = []
    best = 0
    for i, h in enumerate([*heights, 0]):
        while stack and heights[stack[-1]] > h:
            j = stack.pop()
            left = stack[-1] if stack else -1
            best = max(best, heights[j] * (i - left - 1))
        stack.append(i)
    return best
```

### 逐段读懂代码

**最大矩形（对应实现）：**

```python
def largest_rectangle(heights):
    stack = []
    best = 0
    for i, h in enumerate([*heights, 0]):
        while stack and heights[stack[-1]] > h:
            j = stack.pop()
            left = stack[-1] if stack else -1
            best = max(best, heights[j] * (i - left - 1))
        stack.append(i)
    return best
```

- `enumerate([*heights,0])`：用星号解包再拼一个 0，一行完成"复制数组并追加哨兵"。哨兵高度 0 比任何非负柱都矮，保证收尾时全员结算。
- `while stack and heights[stack[-1]]>h`：只弹**严格高于**当前柱的。相等的不弹，这在矩形题是安全的——相等的柱子最后由后弹出的那根覆盖同样的更宽区间，最优值不会丢。
- `left=stack[-1] if stack else -1`：弹出 j 后露出新栈顶，它就是左侧最近的"不高于 j"的柱；栈已空说明 j 左边一路畅通，用 -1 表示虚拟边界。
- `heights[j]*(i-left-1)`：高度乘宽度，宽度两头各让一格（left 和 i 都不可用）。
- `stack.append(i)` 无条件压栈；注意哨兵下标也会进栈，但循环随即结束，不会有人去读 `heights[哨兵下标]`。
- 循环里没有 `return`，最优值 `best` 在扫描中不断更新，最后统一返回。

**子数组最小值之和（对应实现）：**

```python
def sum_subarray_mins(nums, mod=10 ** 9 + 7):
    stack = []
    total = 0
    for i in range(len(nums) + 1):
        while stack and (i == len(nums) or nums[stack[-1]] >= nums[i]):
            j = stack.pop()
            left = stack[-1] if stack else -1
            total += nums[j] * (j - left) * (i - j)
        stack.append(i)
    return total % mod
```

- `for i in range(len(nums)+1)`：多扫一步让 i 取到 n，配合括号里的 `i==len(nums)` 充当哨兵条件——这一步为真时无条件弹栈，清算所有剩余元素，不需要真的往数组里塞值。
- 弹栈条件用的是 `>=`（对比矩形题的 `>`）：相等也弹。这一字之差就是"最右出现者获得归属"的实现——相等的前一个被后一个弹掉时，right 边界 i 是"小于等于"的第一个位置。
- 于是 left（弹出后的栈顶）必然严格更小（相等的早被弹走了），一侧非严格（右）+ 一侧严格（左），重复最小值不会算两次也不会漏算。
- `total+=nums[j]*(j-left)*(i-j)`：三因子相乘就是"值 × 左端点选法 × 右端点选法"。任何左端点在 (left, j]、右端点在 [j, i) 的子数组，最小值都由 j 负责。
- `return total%mod`：取模放在最后一步，中间用整数运算不丢精度；`mod=10**9+7` 是带默认值的参数，小测试时结果天然小于模数。为什么取模放最后没问题？Python 的大整数不会溢出，先把真实总和全加完再取一次模，比"每步都取模"更省事且结果一致（同余的性质）；换成 C++ 这类会溢出的语言就得边加边取模。
- 对比两个函数的一处细节差异：矩形题弹栈条件是**严格** `>`，最小值题是**非严格** `>=`。这不是笔误——矩形题里相等柱子的面积归属不影响最优值（后弹的那根覆盖同样的更宽区间），怎么写都能对；最小值题是精确计数，多算一次少算一次都会错，必须用"一侧严格一侧非严格"锁死归属。读懂这个差异，你就理解了"正确性要求"和"最优值要求"的差别。

## 5. 分段实现：按顺序运行

**本章接口：** `largest_rectangle(heights)`、`sum_subarray_mins(nums)`

### largest_rectangle

In [1]:
def largest_rectangle(heights):
    stack = []
    best = 0
    for i, h in enumerate([*heights, 0]):
        while stack and heights[stack[-1]] > h:
            j = stack.pop()
            left = stack[-1] if stack else -1
            best = max(best, heights[j] * (i - left - 1))
        stack.append(i)
    return best

### sum_subarray_mins

In [2]:
def sum_subarray_mins(nums, mod=10 ** 9 + 7):
    stack = []
    total = 0
    for i in range(len(nums) + 1):
        while stack and (i == len(nums) or nums[stack[-1]] >= nums[i]):
            j = stack.pop()
            left = stack[-1] if stack else -1
            total += nums[j] * (j - left) * (i - j)
        stack.append(i)
    return total % mod

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[3,1,2,4]; stack=[]; rows=[]
for i in range(len(a)+1):
    while stack and (i==len(a) or a[stack[-1]]>=a[i]):
        j=stack.pop(); left=stack[-1] if stack else -1
        rows.append((j,a[j],left,i,(j-left)*(i-j),a[j]*(j-left)*(i-j)))
    stack.append(i)
show_table(['负责索引','值','左严格小边界','右小于等于边界','区间数','贡献'],rows)

负责索引,值,左严格小边界,右小于等于边界,区间数,贡献
0,3,-1,1,1,3
3,4,2,4,1,4
2,2,1,4,2,4
1,1,-1,4,6,6


## 7. 正确性、适用条件与复杂度

每个弹出索引的左右障碍界定了它可担任最低值的最大范围。最小值计数规则把重复最小值交给最右出现者，左端有j−left种、右端有i−j种选择。

**代价：** O(n)时间、O(n)栈空间；直方图高度非负；最小值和默认按10⁹+7取模。

### 展开理解

- **矩形题为什么对**：每个下标都会在"第一根更矮的柱出现"或哨兵到来时被弹出，弹出那一刻 i 恰好是它的右边界；而栈里压在它下面的正好是一路留下来的"不比它高的柱"，其中最近的那根就是左边界。所以每次结算算出的都是"以 j 为最矮柱能撑起的最大宽度"，把 n 根柱子各当一次最矮柱，所有可能的矩形都考察过了，最优必然被取到。等高的柱子看似归属含糊，但它们中被弹得最晚的那根结算时会覆盖全部宽度，面积不会少算。
- **最小值和为什么对**：任何子数组都有唯一的最小值负责人——如果最小值出现多次，就取最右边那次。左边界用严格小于保证：从 left 往右到 j 之间没有更小或相等的（相等的已被弹走），所以子数组左端点只要落在 (left, j] 就不会把更小的元素卷进来；右边界用小于等于保证：[j, i) 之间没有比 nums[j] 更小的，相等的那次出现也归 j 管。每个子数组被且只被一个 j 计数一次，总和自然正确。
- **复杂度**：两个函数里每个下标最多入栈一次、出栈一次，while 的总弹出次数是 n，所以都是 O(n) 时间、O(n) 栈空间。感受一下量级：n=10⁵ 时大约二十万次栈操作，毫秒级；而暴力枚举子数组是 O(n²)≈10¹⁰ 量级，分钟级都跑不完。前提（“正确性与复杂度”部分）：直方图高度非负；结果按 10⁹+7 取模。
- **一个常见疑问：均摊为什么成立？** 你可能担心"某一步 while 连弹很多次，会不会让复杂度爆炸"。答案是不会：弹出的每个元素都是从栈里来的，而栈里的元素都是之前某一步压进去的——**弹出的总次数不可能超过压入的总次数 n**。最坏形状（比如单调递减的柱子）是把所有弹出攒到最后哨兵来时一口气做完，总数仍是 n，只是分布不均匀。

同样给这一章列一份踩坑清单，每条都能在上面的测试或穷举里找到对应的"照妖镜"：

- 矩形题忘了拼哨兵 0：栈里残留的柱子永远不被结算，递增数组（如 `[1,2,3]`）直接算错。
- 宽度写成 `i-left` 而不是 `i-left-1`：把边界柱自己的宽度也数进去，面积偏大。
- 贡献公式把 `(j-left)*(i-j)` 写反或漏一个因子：乘法交换律救不了漏乘。
- 最小值题把弹栈条件写成 `>`（两侧都严格）：`[1,1]` 这类重复值数组会重计，穷举立刻失败。
- 左边界在弹出**前**取栈顶：取到的是自己或更近的错误元素，必须先 pop 再看新栈顶。
- 矩形题和最小值题的两套比较符（`>` vs `>=`）互相抄混：一个求最优（宽松没事）、一个求精确计数（严格有事），不能照搬。

## 8. 单元测试：每个用例在检查什么

这一章的两条手写断言都是 LeetCode 的官方示例，价值在"形状"；真正兜底的是后面的上千条穷举。

- `assert largest_rectangle([2,1,5,6,2,3])==10`：LC84 的经典用例。覆盖"高峰被两侧低谷夹住"（5、6 → 10）、"矮柱靠宽度取胜"（1×6=6 不敌 10）与哨兵清算三条路径。
- `assert sum_subarray_mins([3,1,2,4])==17`：LC907 的经典用例，10 个子数组的最小值手算和恰为 17（第一节列过明细）。
- 双重 for 穷举（`n in range(7)`，值取 {0,1,2}，共 1+3+9+27+81+243+729=1093 个数组）：对每个小数组用两层循环的暴力参照——`min(a[l:r])*(r-l)` 枚举最大矩形、`sum(min(a[l:r]))` 枚举最小值和——逐一比对。**值域刻意只有 0~2**，这让"相等元素"的碰撞极其频繁，正好反复拷打"一侧严格、一侧非严格"的重复值归属规则：如果两边都严格（重计）或都不严格（漏计），这上千条穷举里必然有对不上的。注意参照式没有取模，小数组结果小于 10⁹+7，两种口径天然一致。
- 参照代码本身值得读一遍：`max((min(a[l:r])*(r-l) ...), default=0)` 里的 `default=0` 是为空数组（n=0）准备的——没有子区间时 max 无从取值，直接给 0。这类小细节就是"暴力直译题意"的写法，拿它当规格书去核对优化版最合适。

In [4]:
assert largest_rectangle([2, 1, 5, 6, 2, 3]) == 10

assert sum_subarray_mins([3, 1, 2, 4]) == 17

from itertools import product

for n in range(7):
    for a in product([0, 1, 2], repeat=n):
        best = max((min(a[l:r]) * (r - l) for l in range(n) for r in range(l + 1, n + 1)), default=0)
        total = sum((min(a[l:r]) for l in range(n) for r in range(l + 1, n + 1)))
        assert largest_rectangle(a) == best and sum_subarray_mins(a) == total

print('N055: 所有本章断言通过')

N055: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 推导子数组最大值和。

**练习 2：** 构造两侧同时严格造成重复/漏计的案例。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（子数组最大值和）**：提示：把"最小"整体镜像成"最大"——栈从存"未找到更小的候选"变成存"未找到更大的候选"，弹栈条件里的 `>=` 翻成 `<=`（右边界取大于等于），left 的含义变成"最近的严格更大者"。贡献公式形状不变：`nums[j]*(j-left)*(i-j)`，相等的最大值归最右出现者。手算 `[2,1,3]` 验证：子数组最大值分别是 2,2,3,1,3,3，和=14。
- **练习 2（两侧同时严格会怎样）**：提示：用 `[1,1]` 做最小白鼠。正确答案是 3（子数组 [1]、[1]、[1,1] 各贡献 1）。如果你把右边界也改成严格小于（弹栈条件 `>` 而不是 `>=`），两个 1 都会被哨兵弹出、各自宣称管辖全区间，中间的 [1,1] 被数两次，得 4——重复；反过来如果左边界也取非严格，跨过两端的子数组没人认领，得 2——漏。把这两种错误版本各写一遍、和暴力对答案，你会真正记住"一侧严格、一侧非严格"不是玄学而是防重防漏的机制。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def largest_rectangle(heights):
    stack = []
    best = 0
    for i, h in enumerate([*heights, 0]):
        while stack and heights[stack[-1]] > h:
            j = stack.pop()
            left = stack[-1] if stack else -1
            best = max(best, heights[j] * (i - left - 1))
        stack.append(i)
    return best

def sum_subarray_mins(nums, mod=10 ** 9 + 7):
    stack = []
    total = 0
    for i in range(len(nums) + 1):
        while stack and (i == len(nums) or nums[stack[-1]] >= nums[i]):
            j = stack.pop()
            left = stack[-1] if stack else -1
            total += nums[j] * (j - left) * (i - j)
        stack.append(i)
    return total % mod

# 示例与回归测试
assert largest_rectangle([2, 1, 5, 6, 2, 3]) == 10

assert sum_subarray_mins([3, 1, 2, 4]) == 17

from itertools import product

for n in range(7):
    for a in product([0, 1, 2], repeat=n):
        best = max((min(a[l:r]) * (r - l) for l in range(n) for r in range(l + 1, n + 1)), default=0)
        total = sum((min(a[l:r]) for l in range(n) for r in range(l + 1, n + 1)))
        assert largest_rectangle(a) == best and sum_subarray_mins(a) == total

print('N055: 所有本章断言通过')

N055: 所有本章断言通过


## 11. 代表题与迁移

- **84. Largest Rectangle in Histogram**：练"弹栈瞬间定右边界、栈顶定左边界"与哨兵清算，`largest_rectangle` 是本题的标准解。
- **907. Sum of Subarray Minimums**：练贡献法三因子公式与重复值的去重归属，`sum_subarray_mins` 直接对应（含 10⁹+7 取模）。
- **42. Trapping Rain Water**（对照题）：同样靠"左右最近更高柱"决定每个位置的水位，是本章"左右作用范围"思想在另一道题上的变形，值得对照思考。

一句话总结本章的迁移路线：只要统计量能拆成"每个元素 × 它负责的区间个数/宽度"，就能考虑单调栈 + 贡献法；先问"谁是瓶颈、瓶颈管多宽"，再写代码。

下一章（N056）把栈换成两头能弹的队列，"管辖范围"的思想不变，但候选还要看保质期——建议带着这个对比去读。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。